# Causality

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import warnings
warnings.simplefilter("ignore")

In [ ]:
# polars
import polars as pl
from IPython.display import display
import numpy as np

def _append(base, extra):
    """Append a value or a Series onto a Series."""
    extra_s = extra if isinstance(extra, pl.Series) else pl.Series([extra])
    if base.len() == 0:
        return extra_s
    if base.dtype != extra_s.dtype:
        try:
            extra_s = extra_s.cast(base.dtype, strict=False)
        except Exception:
            base = base.cast(extra_s.dtype, strict=False)
    return pl.concat([base, extra_s])


## Review: Comparing Two Samples

In [ ]:
# datascience
births = Table.read_table('baby.csv')

In [ ]:
# polars
births_pl = pl.read_csv('baby.csv', eol_char='\r')

In [ ]:
# datascience
means = births.group('Maternal Smoker', np.average)
means

In [ ]:
# polars
means_pl = (lambda _df, _k='Maternal Smoker': _df.group_by(_k).agg([pl.col(c).mean().alias(f"{c} average") for c in _df.columns if c != _k and _df[c].dtype.is_numeric()]).sort(_k))(births_pl)
means_pl

In [ ]:
# datascience
observed_difference = means.column(1).item(1) - means.column(1).item(0)
observed_difference

In [ ]:
# polars
observed_difference_pl = means_pl.to_series(1).item(1) - means_pl.to_series(1).item(0)
observed_difference_pl

In [ ]:
# datascience
def difference_of_means(table, numeric_label, group_label):
    """
    Takes: name of table, column label of numerical variable,
    column label of group-label variable
    
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = table.select(numeric_label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
# polars
def difference_of_means_pl(table, numeric_label, group_label):
    """
    Takes: name of table, column label of numerical variable,
    column label of group-label variable
    
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = table.select(numeric_label, group_label)  
    
    # table containing group means
    means_table = (lambda _df, _k=group_label: _df.group_by(_k).agg([pl.col(c).mean().alias(f"{c} average") for c in _df.columns if c != _k and _df[c].dtype.is_numeric()]).sort(_k))(reduced)
    
    # array of group means
    means_pl = means_table.to_series(1)
    
    return means_pl.item(1) - means_pl.item(0)

In [ ]:
# datascience
def one_simulated_difference(table, numeric_label, group_label):
    """
    Takes: name of table, column label of numerical variable,
    column label of group-label variable
    
    Returns: Difference of means of the two groups after shuffling labels
    """
    
    # array of shuffled labels
    shuffled_labels = table.sample(
        with_replacement = False).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(numeric_label).with_column(
        'Shuffled Label', shuffled_labels)
    
    return difference_of_means(
        shuffled_table, numeric_label, 'Shuffled Label')   

In [ ]:
# polars
def one_simulated_difference_pl(table, numeric_label, group_label):
    """
    Takes: name of table, column label of numerical variable,
    column label of group-label variable
    
    Returns: Difference of means of the two groups after shuffling labels
    """
    
    # array of shuffled labels
    shuffled_labels = (lambda _df: _df.sample(n=_df.height, with_replacement=False, shuffle=True))(table).get_column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(numeric_label).with_columns(pl.Series('Shuffled Label', shuffled_labels))
    
    return difference_of_means_pl(
        shuffled_table, numeric_label, 'Shuffled Label')   

In [ ]:
# datascience
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# datascience
differences = make_array()

for i in np.arange(2000):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
# polars
differences_pl = []

for i in pl.int_range(2000, eager=True):
    new_difference = one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')
    differences_pl.append(new_difference)
differences_pl = pl.Series("values", differences_pl)

In [ ]:
# datascience
Table().with_column('Difference Between Group Means', differences).hist()
plots.ylim(-.02, .4)
plots.scatter(observed_difference, 0, color='red')

plots.title('Prediction Under the Null Hypothesis');

In [ ]:
# polars
# matplotlib
_df = pl.DataFrame({'Difference Between Group Means': differences_pl})
_cols = [c for c in _df.columns if _df[c].dtype.is_numeric()]
if len(_cols) == 1:
    plots.hist(_df.get_column(_cols[0]).to_numpy(), density=True)
    plots.xlabel(_cols[0])
else:
    for _col in _cols:
        plots.hist(_df.get_column(_col).to_numpy(), density=True, label=_col)
    plots.legend()
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))
plots.ylim(-.02, .4)
plots.scatter(observed_difference_pl, 0, color='red')

plots.title('Prediction Under the Null Hypothesis');

## Randomized Control Experiment

In [ ]:
# datascience
botox = Table.read_table('bta.csv')
botox.show()

In [ ]:
# polars
botox_pl = pl.read_csv('bta.csv')
with pl.Config(tbl_rows=-1):
    display(botox_pl)

**DISCUSS**: How would you use table methods to summarize the `botox` table?

In [ ]:
# datascience
botox.pivot('Result', 'Group')

In [ ]:
# polars
(lambda _p: _p.select(['Group'] + sorted([c for c in _p.columns if c != 'Group'], key=str)))(botox_pl.pivot('Result', index='Group', values='Result', aggregate_function='len').sort('Group'))

In [ ]:
# datascience
botox.group('Group', np.average)

In [ ]:
# polars
(lambda _df, _k='Group': _df.group_by(_k).agg([pl.col(c).mean().alias(f"{c} average") for c in _df.columns if c != _k and _df[c].dtype.is_numeric()]).sort(_k))(botox_pl)

### Testing the Hypothesis

In [ ]:
# datascience
observed_diff = difference_of_means(botox, 'Result', 'Group')
observed_diff

In [ ]:
# polars
observed_diff_pl = difference_of_means_pl(botox_pl, 'Result', 'Group')
observed_diff_pl

In [ ]:
# datascience
one_simulated_difference(botox, 'Result', 'Group')

In [ ]:
# polars
one_simulated_difference_pl(botox_pl, 'Result', 'Group')

In [ ]:
# datascience
simulated_diffs = make_array()

for i in np.arange(10000):
    sim_diff = one_simulated_difference(botox, 'Result', 'Group')
    simulated_diffs = np.append(simulated_diffs, sim_diff)

In [ ]:
# polars
simulated_diffs_pl = []

for i in pl.int_range(10000, eager=True):
    sim_diff = one_simulated_difference_pl(botox_pl, 'Result', 'Group')
    simulated_diffs_pl.append(sim_diff)
simulated_diffs_pl = pl.Series("values", simulated_diffs_pl)

In [ ]:
# datascience
col_name = 'Distances between groups'
Table().with_column(col_name, simulated_diffs).hist(col_name)

plots.ylim(-.02, 2.3)
plots.scatter(observed_diff, 0, color='red');

In [ ]:
# polars
# matplotlib
col_name_pl = 'Distances between groups'
_df = pl.DataFrame({col_name_pl: simulated_diffs_pl})
plots.hist(_df.get_column(col_name_pl).to_numpy(), density=True)
plots.xlabel(col_name_pl)
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

plots.ylim(-.02, 2.3)
plots.scatter(observed_diff_pl, 0, color='red');

In [ ]:
# datascience
# compute p-value
sum(simulated_diffs >= observed_diff) / len(simulated_diffs)

In [ ]:
# polars
# compute p-value
sum(simulated_diffs_pl >= observed_diff_pl) / len(simulated_diffs_pl)

**AT HOME**: Compare the null/alternative hypotheses in lecture vs. textbook (Section 12.2)